# FD-21: M2 v1 (microsoft/unixcoder-base @1024) — Kaggle 2xT4
Шаги workflow R10: пилот 100 шагов → overfit 200x20 → полный прогон 3 эпохи → усечение → предсказания → оценка. Выбор эпохи — по PR-AUC **val_real**. Подробнее — `docs/kaggle_fd21_guide.md`.

In [ ]:
%cd /kaggle/working
import os
from kaggle_secrets import UserSecretsClient
try:
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    _repo = f"https://{_tok}@github.com/o1uh/FreshDoc.git"
    print("clone: private (token)")
except Exception:
    _repo = "https://github.com/o1uh/FreshDoc.git"
    print("clone: public fallback (repo must be public)")
!if [ ! -d FreshDoc ]; then git clone --quiet {_repo} > /tmp/clone.log 2>&1 || (echo "CLONE FAILED:"; sed -E 's#(https?://)[^@]+@#\1***@#g' /tmp/clone.log); fi
del _tok, _repo  # не светить токен дальше
%cd /kaggle/working/FreshDoc
!git checkout feat/FD-21-m2-train
!git pull --ff-only origin feat/FD-21-m2-train || echo "PULL SKIPPED: убедись, что код свежий"
!pip install -q -r requirements.txt
import os
os.environ['NCCL_P2P_DISABLE'] = '1'
!nvidia-smi

In [ ]:
import pandas as pd
_p = "/kaggle/input/freshdoc-dataset/dataset_v1.parquet"
_df = pd.read_parquet(_p, columns=["split"])
assert len(_df) == 27203, f"dataset rows {len(_df)} != 27203 (старый файл без gold-патча?)"
_exp = {"train": 21997, "val": 559, "test_real": 2895, "test_synth": 1552, "gold": 200}
_got = _df["split"].value_counts().to_dict()
assert _got == _exp, f"split mismatch: {_got}"
print("dataset OK:", _got)
del _df


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
for m in ['microsoft/unixcoder-base']:
    tok = AutoTokenizer.from_pretrained(m)
    mod = AutoModel.from_pretrained(m).to('cuda')
    out = mod(**tok('def test(): pass', return_tensors='pt').to('cuda'))
    print(m, tuple(out.last_hidden_state.shape))
print('smoke OK')
del mod, out, tok
import gc; gc.collect()
torch.cuda.empty_cache()
print("GPU freed")


In [ ]:
!python scripts/train_m2.py --config config/train.yaml --data_path /kaggle/input/freshdoc-dataset/dataset_v1.parquet --output_dir /kaggle/working/checkpoints/m2_pilot --pilot_steps 100

In [ ]:
!python scripts/train_m2.py --config config/train.yaml --data_path /kaggle/input/freshdoc-dataset/dataset_v1.parquet --output_dir /kaggle/working/checkpoints/m2_overfit --overfit_test

In [ ]:
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml --data_path /kaggle/input/freshdoc-dataset/dataset_v1.parquet --output_dir /kaggle/working/checkpoints/m2_run1

In [ ]:
!python scripts/check_truncation.py --model microsoft/unixcoder-base --max_len 1024 --dataset /kaggle/input/freshdoc-dataset/dataset_v1.parquet --out /kaggle/working/m2_truncation.json
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/m2_run1/best --dataset /kaggle/input/freshdoc-dataset/dataset_v1.parquet --out /kaggle/working/m2.csv
!python scripts/baselines_m0_m1.py --dataset /kaggle/input/freshdoc-dataset/dataset_v1.parquet --out_m0 /kaggle/working/m0.csv --out_m1 /kaggle/working/m1.csv --out_model /kaggle/working/m1_hgb.joblib --out_report /kaggle/working/m1_report.json
!python scripts/evaluate.py --dataset /kaggle/input/freshdoc-dataset/dataset_v1.parquet --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/m2.csv --out_metrics /kaggle/working/metrics_m2.json --out_table /kaggle/working/e1_m2.csv